# Exploring Our Bronze Data

Before we clean anything, let's sit down with the data and understand what we're working with.

We have **4 types** of trip data and **1 reference file** — all loaded as-is into Bronze Delta tables.

Let's go through them **one by one**, see what columns they have, what the values look like, and spot anything that doesn't look right.

## 1. What tables do we have?

Let's start by simply listing everything in our Bronze layer.

In [0]:
CATALOG = "nyc_taxi_lakehouse"
SCHEMA  = "bronze"

tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()

print(f"We have {len(tables)} tables in {CATALOG}.{SCHEMA}:\n")
for t in tables:
    count = spark.table(f"{CATALOG}.{SCHEMA}.{t.tableName}").count()
    print(f"  • {t.tableName:40s} → {count:>12,} rows")

---

## 2. Taxi Zone Lookup (Our Reference Table)

This is the small table that tells us **where** each trip happened. Every trip has a pickup and dropoff LocationID — this table maps those IDs to real zone names.

Let's take a look.

In [0]:
zones = spark.table(f"{CATALOG}.{SCHEMA}.taxi_zone_lookup")
zones.display()


### What are the columns?

| Column | What it means |
|--------|--------------|
| **LocationID** | A unique number (1–265) assigned to each taxi zone in NYC |
| **Borough** | The borough the zone belongs to (Manhattan, Brooklyn, Queens, Bronx, Staten Island, EWR) |
| **Zone** | The actual neighborhood name (e.g., "Upper East Side North") |
| **service_zone** | What kind of taxi service operates there — Yellow Zone (Manhattan core), Boro Zone (outer boroughs), Airports, or EWR |

### Quick check — any missing values?

In [0]:
from pyspark.sql.functions import col, count, when, sum as spark_sum

zone_cols = zones.columns
zone_nulls = zones.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) 
    for c in zone_cols
])

print("Missing values per column:")
zone_nulls.display()


### How are the boroughs distributed?

In [0]:
zones.groupBy("Borough").count().orderBy("count", ascending=False).display()

### How are the service zones distributed?

In [0]:
zones.groupBy("service_zone").count().orderBy("count", ascending=False).display()

### Zone Lookup — Summary

✅ Very clean — only 1-2 nulls in Borough/Zone/service_zone (likely the "Unknown" zone 264).

This table is our **dimension table** for location. Every trip's pickup/dropoff LocationID joins here.

---

## 3. Yellow Taxi

The classic NYC yellow cab. Let's look at one month to understand the structure, then we'll check if both months have the same shape.

In [0]:
yellow = spark.table(f"{CATALOG}.{SCHEMA}.yellow_tripdata_2024_10")

print("Schema:")
yellow.printSchema()

### Let's see some actual rows

In [0]:
yellow.limit(10).display()

### What does each column mean?

| Column | Meaning |
|--------|---------|
| **VendorID** | Which meter system recorded the trip (1 = Creative Mobile Technologies, 2 = VeriFone Inc.) |
| **tpep_pickup_datetime** | When the meter was engaged (trip started) |
| **tpep_dropoff_datetime** | When the meter was disengaged (trip ended) |
| **passenger_count** | Number of passengers — driver-entered |
| **trip_distance** | Distance in miles reported by the taximeter |
| **RatecodeID** | Rate code: 1=Standard, 2=JFK, 3=Newark, 4=Nassau/Westchester, 5=Negotiated, 6=Group ride |
| **store_and_fwd_flag** | Y = trip was held in vehicle memory before sending (no server connection), N = sent immediately |
| **PULocationID** | Pickup taxi zone → joins to zone lookup |
| **DOLocationID** | Dropoff taxi zone → joins to zone lookup |
| **payment_type** | 1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown |
| **fare_amount** | The time-and-distance fare from the meter |
| **extra** | Extras and surcharges ($0.50 rush hour, $1 overnight, etc.) |
| **mta_tax** | $0.50 MTA tax triggered automatically |
| **tip_amount** | Tip — **only recorded for credit card payments** |
| **tolls_amount** | All tolls paid during the trip |
| **improvement_surcharge** | $1.00 improvement surcharge |
| **total_amount** | Total charged to passenger (does NOT include cash tips) |
| **congestion_surcharge** | $2.50 for trips in the congestion zone |
| **Airport_fee** | $1.75 for pickups at LaGuardia and JFK |

### How's the data quality? Let's check column by column.

In [0]:
print(f"Total rows: {yellow.count():,}\n")

# Missing values
print("── Missing Values ──")
for c in yellow.columns:
    null_count = yellow.where(col(c).isNull()).count()
    if null_count > 0:
        pct = (null_count / yellow.count()) * 100
        print(f"  {c:30s} → {null_count:>10,} nulls ({pct:.2f}%)")


### Are there negative fares? That seems wrong...

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max, avg, round as spark_round

money_columns = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount", 
                  "improvement_surcharge", "total_amount", "congestion_surcharge", "Airport_fee"]

print("── Financial Columns: Min / Avg / Max ──\n")
for c in money_columns:
    stats = yellow.select(
        spark_min(col(c)).alias("min"),
        spark_round(avg(col(c)), 2).alias("avg"),
        spark_max(col(c)).alias("max"),
        spark_sum(when(col(c) < 0, 1).otherwise(0)).alias("negative_count")
    ).collect()[0]
    
    flag = " ← ⚠ NEGATIVES!" if stats["negative_count"] > 0 else ""
    print(f"  {c:25s}  min: {stats['min']:>10}  avg: {stats['avg']:>10}  max: {stats['max']:>10}  negatives: {stats['negative_count']:>8,}{flag}")


### What about trip distance and passenger count?

In [0]:
print("── Trip Distance ──")
dist_stats = yellow.select(
    spark_min("trip_distance").alias("min"),
    spark_round(avg("trip_distance"), 2).alias("avg"),
    spark_max("trip_distance").alias("max"),
    spark_sum(when(col("trip_distance") == 0, 1)).alias("zero_count"),
    spark_sum(when(col("trip_distance") > 100, 1)).alias("over_100_miles")
).collect()[0]

print(f"  Min: {dist_stats['min']}  Avg: {dist_stats['avg']}  Max: {dist_stats['max']}")
print(f"  Zero distance trips: {dist_stats['zero_count']:,}")
print(f"  Over 100 miles: {dist_stats['over_100_miles']:,}")

print("\n── Passenger Count ──")
pass_stats = yellow.select(
    spark_sum(when(col("passenger_count") == 0, 1)).alias("zero"),
    spark_sum(when(col("passenger_count").isNull(), 1)).alias("null"),
    spark_sum(when(col("passenger_count") > 6, 1)).alias("over_6")
).collect()[0]

print(f"  Zero passengers: {pass_stats['zero']:,}")
print(f"  Null passengers: {pass_stats['null']:,}")
print(f"  Over 6 passengers: {pass_stats['over_6']:,}")

### Are there any trips with dates outside October 2024?

In [0]:
from pyspark.sql.functions import month, year

print("── Pickup dates that fall outside the expected month ──")
yellow.groupBy(year("tpep_pickup_datetime").alias("year"), month("tpep_pickup_datetime").alias("month")) \
      .count() \
      .orderBy("year", "month") \
      .display()


### Does the second month (November) have the same structure?

In [0]:
yellow_nov = spark.table(f"{CATALOG}.{SCHEMA}.yellow_tripdata_2024_11")

oct_cols = set(yellow.columns)
nov_cols = set(yellow_nov.columns)

if oct_cols == nov_cols:
    print("✅ Both months have the exact same columns — good!")
else:
    print("⚠ Columns differ!")
    print(f"  Only in October: {oct_cols - nov_cols}")
    print(f"  Only in November: {nov_cols - oct_cols}")


---

## 4. Green Taxi

Green taxis serve the outer boroughs and northern Manhattan — areas where yellow cabs are less common. Let's see how their data compares.

In [0]:
green = spark.table(f"{CATALOG}.{SCHEMA}.green_tripdata_2024_10")

print("Schema:")
green.printSchema()

In [0]:
green.limit(10).display()

### What does each column mean?

Most columns are the **same as yellow taxi**, with a few differences:

| Column | Meaning | Same as Yellow? |
|--------|---------|-----------------|
| **lpep_pickup_datetime** | Pickup timestamp | Same concept, different name (lpep vs tpep) |
| **lpep_dropoff_datetime** | Dropoff timestamp | Same concept, different name |
| **ehail_fee** | E-hail fee | ❌ **Green only** — but 100% null (deprecated) |
| **trip_type** | 1 = Street-hail, 2 = Dispatch | ❌ **Green only** — how the trip was initiated |
| **payment_type** | Same codes as yellow | ✅ Same but stored as float64 instead of int64 |

Everything else (fare, distance, locations, etc.) is the same as yellow taxi.

### Missing values?

In [0]:
print(f"Total rows: {green.count():,}\n")

print("── Missing Values ──")
for c in green.columns:
    null_count = green.where(col(c).isNull()).count()
    if null_count > 0:
        pct = (null_count / green.count()) * 100
        print(f"  {c:30s} → {null_count:>10,} nulls ({pct:.2f}%)")


### Financial column ranges — same check as yellow

In [0]:
green_money = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
               "improvement_surcharge", "total_amount", "congestion_surcharge"]

print("── Financial Columns: Min / Avg / Max ──\n")
for c in green_money:
    stats = green.select(
        spark_min(col(c)).alias("min"),
        spark_round(avg(col(c)), 2).alias("avg"),
        spark_max(col(c)).alias("max"),
        spark_sum(when(col(c) < 0, 1).otherwise(0)).alias("negative_count")
    ).collect()[0]
    
    flag = " ← ⚠ NEGATIVES!" if stats["negative_count"] > 0 else ""
    print(f"  {c:25s}  min: {stats['min']:>10}  avg: {stats['avg']:>10}  max: {stats['max']:>10}  negatives: {stats['negative_count']:>8,}{flag}")


### Trip distance and passengers

In [0]:
print("── Trip Distance ──")
dist_stats = green.select(
    spark_min("trip_distance").alias("min"),
    spark_round(avg("trip_distance"), 2).alias("avg"),
    spark_max("trip_distance").alias("max"),
    spark_sum(when(col("trip_distance") == 0, 1)).alias("zero_count"),
    spark_sum(when(col("trip_distance") > 100, 1)).alias("over_100_miles")
).collect()[0]

print(f"  Min: {dist_stats['min']}  Avg: {dist_stats['avg']}  Max: {dist_stats['max']}")
print(f"  Zero distance trips: {dist_stats['zero_count']:,}")
print(f"  Over 100 miles: {dist_stats['over_100_miles']:,}")

print("\n── Passenger Count ──")
pass_stats = green.select(
    spark_sum(when(col("passenger_count") == 0, 1)).alias("zero"),
    spark_sum(when(col("passenger_count").isNull(), 1)).alias("null"),
    spark_sum(when(col("passenger_count") > 6, 1)).alias("over_6")
).collect()[0]

print(f"  Zero passengers: {pass_stats['zero']:,}")
print(f"  Null passengers: {pass_stats['null']:,}")
print(f"  Over 6 passengers: {pass_stats['over_6']:,}")

### Does the second month have the same structure?

In [0]:
green_nov = spark.table(f"{CATALOG}.{SCHEMA}.green_tripdata_2024_11")

if set(green.columns) == set(green_nov.columns):
    print("✅ Both months have the exact same columns — good!")
else:
    print("⚠ Columns differ!")

---

## 5. For-Hire Vehicles (FHV)

These are traditional black cars and livery vehicles — **not** Uber/Lyft. This dataset is much simpler (fewer columns) and unfortunately much less complete.

In [0]:
fhv = spark.table(f"{CATALOG}.{SCHEMA}.fhv_tripdata_2024_10")

print("Schema:")
fhv.printSchema()


In [0]:
fhv.limit(10).display()

### What does each column mean?

| Column | Meaning |
|--------|---------|
| **dispatching_base_num** | TLC base license number of the company that dispatched the trip |
| **pickup_datetime** | When the trip started |
| **dropOff_datetime** | When the trip ended — ⚠ notice the camelCase (inconsistent with other datasets) |
| **PUlocationID** | Pickup zone — ⚠ notice lowercase 'l' (PUlocationID vs PULocationID in taxi data) |
| **DOlocationID** | Dropoff zone — ⚠ same casing issue |
| **SR_Flag** | Shared ride flag — 100% null (deprecated) |
| **Affiliated_base_number** | The base the vehicle is affiliated with (may differ from dispatching base) |

**Key differences from taxi data:**
- ❌ **No fare/payment information at all**
- ❌ **No passenger count**
- ❌ **No trip distance**
- ⚠ Column names use different casing than taxi data

### Missing values — this is where FHV gets concerning

In [0]:
print(f"Total rows: {fhv.count():,}\n")

print("── Missing Values ──")
for c in fhv.columns:
    null_count = fhv.where(col(c).isNull()).count()
    if null_count > 0:
        pct = (null_count / fhv.count()) * 100
        print(f"  {c:30s} → {null_count:>10,} nulls ({pct:.2f}%)")


### How many unique dispatch bases are there?

In [0]:
print("── Top 10 Dispatching Bases ──")
fhv.groupBy("dispatching_base_num").count().orderBy("count", ascending=False).limit(10).display()

### Any duplicates?

In [0]:
total = fhv.count()
distinct = fhv.distinct().count()
dupes = total - distinct
print(f"  Total rows:    {total:,}")
print(f"  Distinct rows: {distinct:,}")
print(f"  Duplicates:    {dupes:,} ({(dupes/total*100):.2f}%)")

### Does the second month have the same structure?

In [0]:
fhv_nov = spark.table(f"{CATALOG}.{SCHEMA}.fhv_tripdata_2024_11")

if set(fhv.columns) == set(fhv_nov.columns):
    print("✅ Both months have the exact same columns — good!")
else:
    print("⚠ Columns differ!")

---

## 6. High-Volume For-Hire Vehicles (FHVHV)

This is the **big one** — Uber and Lyft. ~20 million trips per month. It also has the **richest schema** of all four datasets.

In [0]:
fhvhv = spark.table(f"{CATALOG}.{SCHEMA}.fhvhv_tripdata_2024_10")

print("Schema:")
fhvhv.printSchema()

In [0]:
fhvhv.limit(10).display()

### What does each column mean?

| Column | Meaning |
|--------|---------|
| **hvfhs_license_num** | License number: HV0003 = Uber, HV0005 = Lyft |
| **dispatching_base_num** | Base that dispatched the trip |
| **originating_base_num** | Base where the request originated (may differ) |
| **request_datetime** | When the passenger requested the ride |
| **on_scene_datetime** | When the driver arrived at pickup |
| **pickup_datetime** | When the trip actually started |
| **dropoff_datetime** | When the trip ended |
| **PULocationID** | Pickup zone |
| **DOLocationID** | Dropoff zone |
| **trip_miles** | Distance traveled (⚠ called trip_miles here, trip_distance in taxi data) |
| **trip_time** | Duration in **seconds** |
| **base_passenger_fare** | The fare charged to the passenger |
| **tolls** | Toll amount (⚠ called tolls here, tolls_amount in taxi data) |
| **bcf** | Black Car Fund contribution (2.5% of fare + tolls) |
| **sales_tax** | NY State sales tax |
| **congestion_surcharge** | Congestion pricing surcharge |
| **airport_fee** | Airport pickup/dropoff fee |
| **tips** | Tip amount (⚠ called tips here, tip_amount in taxi data) |
| **driver_pay** | What the driver actually received |
| **shared_request_flag** | Y/N — did the passenger request a shared/pool ride? |
| **shared_match_flag** | Y/N — was the ride actually shared with another passenger? |
| **access_a_ride_flag** | Y/N — was this an Access-A-Ride (disability program) trip? |
| **wav_request_flag** | Y/N — was a wheelchair accessible vehicle requested? |
| **wav_match_flag** | Y/N — was a WAV actually provided? |

### Who's bigger — Uber or Lyft?

In [0]:
fhvhv.groupBy("hvfhs_license_num").count().orderBy("count", ascending=False).display()

### Missing values

In [0]:
print(f"Total rows: {fhvhv.count():,}\n")

print("── Missing Values ──")
for c in fhvhv.columns:
    null_count = fhvhv.where(col(c).isNull()).count()
    if null_count > 0:
        pct = (null_count / fhvhv.count()) * 100
        print(f"  {c:30s} → {null_count:>10,} nulls ({pct:.2f}%)")

### Financial columns — any negatives?

In [0]:
fhvhv_money = ["base_passenger_fare", "tolls", "bcf", "sales_tax", 
               "congestion_surcharge", "airport_fee", "tips", "driver_pay"]

print("── Financial Columns: Min / Avg / Max ──\n")
for c in fhvhv_money:
    stats = fhvhv.select(
        spark_min(col(c)).alias("min"),
        spark_round(avg(col(c)), 2).alias("avg"),
        spark_max(col(c)).alias("max"),
        spark_sum(when(col(c) < 0, 1).otherwise(0)).alias("negative_count")
    ).collect()[0]
    
    flag = " ← ⚠ NEGATIVES!" if stats["negative_count"] > 0 else ""
    print(f"  {c:25s}  min: {stats['min']:>10}  avg: {stats['avg']:>10}  max: {stats['max']:>10}  negatives: {stats['negative_count']:>8,}{flag}")


### Trip distance and time

In [0]:
print("── Trip Miles ──")
dist_stats = fhvhv.select(
    spark_min("trip_miles").alias("min"),
    spark_round(avg("trip_miles"), 2).alias("avg"),
    spark_max("trip_miles").alias("max"),
    spark_sum(when(col("trip_miles") == 0, 1)).alias("zero_count"),
    spark_sum(when(col("trip_miles") > 100, 1)).alias("over_100_miles")
).collect()[0]

print(f"  Min: {dist_stats['min']}  Avg: {dist_stats['avg']}  Max: {dist_stats['max']}")
print(f"  Zero distance trips: {dist_stats['zero_count']:,}")
print(f"  Over 100 miles: {dist_stats['over_100_miles']:,}")

print("\n── Trip Time (in seconds) ──")
time_stats = fhvhv.select(
    spark_min("trip_time").alias("min"),
    spark_round(avg("trip_time"), 0).alias("avg"),
    spark_max("trip_time").alias("max"),
    spark_sum(when(col("trip_time") == 0, 1)).alias("zero_count"),
    spark_sum(when(col("trip_time") < 0, 1)).alias("negative_count"),
    spark_sum(when(col("trip_time") > 7200, 1)).alias("over_2_hours")
).collect()[0]

print(f"  Min: {time_stats['min']}s  Avg: {int(time_stats['avg'])}s ({int(time_stats['avg'])//60} min)  Max: {time_stats['max']}s ({time_stats['max']//60} min)")
print(f"  Zero time trips: {time_stats['zero_count']:,}")
print(f"  Negative time: {time_stats['negative_count'] or 0:,}")
print(f"  Over 2 hours: {time_stats['over_2_hours']:,}")


### Shared rides — how common are they?

In [0]:
print("── Shared Ride Requests ──")
fhvhv.groupBy("shared_request_flag").count().orderBy("count", ascending=False).display()

print("── Actually Matched as Shared ──")
fhvhv.groupBy("shared_match_flag").count().orderBy("count", ascending=False).display()

### Does the second month have the same structure?

In [0]:
fhvhv_nov = spark.table(f"{CATALOG}.{SCHEMA}.fhvhv_tripdata_2024_11")

if set(fhvhv.columns) == set(fhvhv_nov.columns):
    print("✅ Both months have the exact same columns — good!")
else:
    print("⚠ Columns differ!")

---

## 7. Comparing Schemas Across All Datasets

Now let's step back and compare. These four datasets describe the same thing — **trips** — but they come from different systems. How different are they really?

In [0]:
print("── Column Count Per Dataset ──")
print(f"  Yellow Taxi:  {len(yellow.columns):>3} columns")
print(f"  Green Taxi:   {len(green.columns):>3} columns")
print(f"  FHV:          {len(fhv.columns):>3} columns")
print(f"  FHVHV:        {len(fhvhv.columns):>3} columns")


### Naming inconsistencies we'll need to fix

Let's see the exact column names side by side for the key fields.

In [0]:
datasets = {
    "Yellow": yellow.columns,
    "Green":  green.columns,
    "FHV":    fhv.columns,
    "FHVHV":  fhvhv.columns,
}

print("── Column Names Per Dataset ──\n")
for name, cols in datasets.items():
    print(f"{name}:")
    for c in sorted(cols):
        print(f"    {c}")
    print()

### Key naming inconsistencies spotted:

| Concept | Yellow | Green | FHV | FHVHV |
|---------|--------|-------|-----|-------|
| Pickup time | tpep_pickup_datetime | lpep_pickup_datetime | pickup_datetime | pickup_datetime |
| Dropoff time | tpep_dropoff_datetime | lpep_dropoff_datetime | dropOff_datetime | dropoff_datetime |
| Pickup zone | PULocationID | PULocationID | PUlocationID | PULocationID |
| Dropoff zone | DOLocationID | DOLocationID | DOlocationID | DOLocationID |
| Distance | trip_distance | trip_distance | *(missing)* | trip_miles |
| Tip | tip_amount | tip_amount | *(missing)* | tips |
| Tolls | tolls_amount | tolls_amount | *(missing)* | tolls |
| Total fare | total_amount | total_amount | *(missing)* | *(base_passenger_fare + fees)* |

---

## 8. Summary of What We Found

### Data Quality Issues (Across All Datasets)

| Issue | Yellow | Green | FHV | FHVHV |
|-------|--------|-------|-----|-------|
| Negative fares | ✅ Yes (many) | ✅ Yes | N/A (no fare data) | ✅ Check above |
| Zero distance trips | ✅ Yes | ✅ Yes (higher %) | N/A | ✅ Check above |
| Missing passenger count | ✅ ~10% | ✅ ~3% | N/A | N/A |
| Missing locations | Minor | Minor | ✅ **78% pickup!** | Minor |
| Future timestamps | ✅ Yes | ✅ Yes | ✅ Yes | ✅ Check above |
| Duplicates | ✅ None | ✅ None | ✅ ~0.15% | ✅ Check above |
| Deprecated columns | None | ehail_fee (100% null) | SR_Flag (100% null) | None |

### Schema Issues

- **Column names differ** across datasets for the same concept
- **Data types differ** (e.g., payment_type is int64 in yellow, float64 in green)
- **FHV is missing** fare, distance, and passenger data entirely

### What Needs To Happen in Silver Layer

1. **Standardize column names** — one consistent naming convention
2. **Remove clearly invalid records** — negative fares, impossible distances
3. **Flag suspicious records** — zero distance, zero passengers
4. **Drop deprecated columns** — ehail_fee, SR_Flag
5. **Deduplicate** — especially FHV data
6. **Enrich with zone lookup** — add borough and zone names
7. **Add derived fields** — trip duration, hour of day, day of week